# 13 - Numeric state and imputation screen

This notebook presents the bounded fixed-model screen for `amount_tsh`, `gps_height`, `population` and `num_private`. It performs no model refits: run `../scripts/run_numeric_screen.py` first to recreate the ignored runtime evidence. The labelled local test, competition predictions and oversampling artefacts remain outside this analysis.

In [1]:
from pathlib import Path
import sys

import pandas as pd

STAGE_DIR = Path.cwd().parent
PROJECT_DIR = STAGE_DIR.parent
SRC_DIR = STAGE_DIR / 'src'
RUNTIME_DIR = PROJECT_DIR / '.runtime' / 'numeric-screen'
DATA_DIR = STAGE_DIR / 'data'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_partitioning import make_cross_validation, partition_modelling_data
from geography_evaluation import make_lga_grouped_partition
from modelling_data import prepare_modelling_data
from numeric_features import fit_geographic_median_model

required = [
    RUNTIME_DIR / 'policy-register.csv',
    RUNTIME_DIR / 'frozen-summary.csv',
    RUNTIME_DIR / 'hybrid-summary.csv',
    RUNTIME_DIR / 'lga-grouped-summary.csv',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Run the numeric screen first; missing={missing!r}')

print(f'Runtime evidence: {RUNTIME_DIR}')

Runtime evidence: C:\_Source\Imperial-ML-AI-Live\Capstone\imperial-capstone\.runtime\numeric-screen


## Audit boundary

The baseline already retains amount magnitude and a recorded-state flag, masks zero height and population before fold-median imputation, retains their missing flags, and passes raw `num_private`. The screen tests only materially different state, removal and geographic-imputation policies. Monotonic logs do not give tree splitters a new ordering, and PCA would densify the mixed sparse feature space without addressing sentinel semantics.

In [2]:
training = pd.read_csv(DATA_DIR / 'TrainingSetValues.csv')
audit_rows = []
for feature in ('amount_tsh', 'gps_height', 'population', 'num_private'):
    values = pd.to_numeric(training[feature], errors='coerce')
    positive = values.loc[values.gt(0)]
    audit_rows.append({
        'feature': feature,
        'zero_rows': int(values.eq(0).sum()),
        'zero_share': values.eq(0).mean(),
        'one_rows': int(values.eq(1).sum()),
        'positive_rows': int(values.gt(0).sum()),
        'distinct_values': int(values.nunique(dropna=False)),
        'positive_median': float(positive.median()),
        'positive_p95': float(positive.quantile(.95)),
    })
audit = pd.DataFrame(audit_rows).set_index('feature')
audit['zero_share'] = audit['zero_share'].map(lambda value: f'{value:.1%}')
display(audit)
measurement_block = (
    training['gps_height'].eq(0)
    & training['population'].eq(0)
    & training['construction_year'].eq(0)
)
print(f'Shared zero-measurement block: {measurement_block.sum():,} rows ({measurement_block.mean():.1%})')

,zero_rows,zero_share,one_rows,positive_rows,distinct_values,positive_median,positive_p95
feature,,,,,,,
amount_tsh,41639,70.1%,3,17761,98,250.0,5000.0
gps_height,20438,34.4%,34,37466,2428,1194.0,1910.0
population,21381,36.0%,7025,38019,1049,150.0,897.0
num_private,58643,98.7%,73,757,65,15.0,102.0


Shared zero-measurement block: 19,668 rows (33.1%)


## Predeclared representations

Every learned median is fitted inside the current training partition. Geographic imputation uses LGA when at least 20 measured rows exist, then region, then the training-partition global median; the original missing-state flag remains visible.

In [3]:
policies = pd.read_csv(RUNTIME_DIR / 'policy-register.csv').set_index('policy')
policies

,label,engineered_features,removed_features,zero_retained,geographic_medians,rationale
policy,,,,,,
baseline,Accepted numeric policy,29,NaN,NaN,NaN,Retain the accepted state flags and fold media...
amount_recorded_only,Amount recorded state only,28,amount_tsh,NaN,NaN,Remove tariff magnitude while retaining amount...
gps_zero_as_value,GPS height zero retained,29,NaN,gps_height,NaN,Retain zero height as a value alongside gps_he...
gps_geographic_median,GPS height geographic median,29,NaN,NaN,gps_height,"Fill zero height from fold-fitted LGA, region ..."
population_zero_as_value,Population zero retained,29,NaN,population,NaN,Retain zero population alongside population_mi...
population_one_state,Population-one state,30,NaN,NaN,NaN,"Expose the 7,025-row population-one spike expl..."
population_geographic_median,Population geographic median,29,NaN,NaN,population,"Fill zero population from fold-fitted LGA, reg..."
num_private_drop,Remove num_private,28,num_private,NaN,NaN,Ablate an undocumented feature that is zero in...
num_private_nonzero_only,num_private non-zero state only,29,num_private,NaN,NaN,Replace sparse magnitude with a non-zero state...


## Frozen-fold screen

Every policy uses the unchanged 55% child-weight-1 depth-8 XGBoost and 45% Random Forest vote. The primary gate requires at least +0.10 percentage points, three fold wins, no fold below -0.25 points and no repair-recall loss beyond two points.

In [4]:
frozen = pd.read_csv(RUNTIME_DIR / 'frozen-summary.csv').set_index('policy')
frozen_display = frozen.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'transformed_features_fold_1', 'passes_gate',
]].copy()
for column in ('mean_accuracy', 'accuracy_change', 'worst_fold_change', 'repair_recall'):
    frozen_display[column] = frozen_display[column].map(lambda value: f'{value:.3%}')
frozen_display

,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,transformed_features_fold_1,passes_gate
policy,,,,,,,,
baseline,Accepted numeric policy,81.625%,0.000%,0,0.000%,34.859%,301,False
population_geographic_median,Population geographic median,81.591%,-0.034%,1,-0.095%,34.656%,300,False
population_one_state,Population-one state,81.589%,-0.036%,3,-0.179%,34.801%,302,False
measurement_block_state,Shared measurement-block state,81.587%,-0.038%,1,-0.116%,34.830%,302,False
num_private_drop,Remove num_private,81.576%,-0.048%,1,-0.105%,34.801%,300,False
population_zero_as_value,Population zero retained,81.553%,-0.072%,1,-0.147%,34.656%,300,False
amount_recorded_only,Amount recorded state only,81.543%,-0.082%,2,-0.253%,35.119%,300,False
num_private_nonzero_only,num_private non-zero state only,81.540%,-0.084%,1,-0.221%,34.714%,301,False
both_geographic_medians,Height and population geographic medians,81.532%,-0.093%,1,-0.168%,34.801%,299,False


The accepted numeric policy leads every challenger. Population geographic median, population-one state and the shared measurement-block state are within 0.04 points but remain below baseline. Retaining zero height or population as a numeric value is clearly worse, supporting their current unavailable-state semantics.

## Geographic population-imputation fallback evidence

In [5]:
modelling_data = prepare_modelling_data(
    training,
    pd.read_csv(DATA_DIR / 'TrainingSetLabels.csv'),
    pd.read_csv(DATA_DIR / 'TestSetValues.csv'),
)
partitioned = partition_modelling_data(modelling_data)

def fallback_counts(name, current_partition, cross_validation):
    rows = []
    for fold, (training_positions, validation_positions) in enumerate(
        cross_validation.split(), start=1
    ):
        training_fold = current_partition.X_development.iloc[training_positions]
        validation_fold = current_partition.X_development.iloc[validation_positions]
        model = fit_geographic_median_model(training_fold, 'population')
        missing = validation_fold['population'].eq(0)
        lga_values = validation_fold.loc[missing, 'lga'].astype('string').str.strip().map(model.lga_medians)
        region_values = validation_fold.loc[missing, 'region'].astype('string').str.strip().map(model.region_medians)
        lga_rows = int(lga_values.notna().sum())
        region_rows = int((lga_values.isna() & region_values.notna()).sum())
        rows.append({
            'design': name, 'fold': fold, 'missing_rows': int(missing.sum()),
            'lga': lga_rows, 'region': region_rows,
            'global': int(missing.sum()) - lga_rows - region_rows,
        })
    return pd.DataFrame(rows)

frozen_fallbacks = fallback_counts(
    'frozen stratified', partitioned, make_cross_validation(partitioned)
)
grouped_partition, grouped_cv, grouped_folds = make_lga_grouped_partition(partitioned)
grouped_fallbacks = fallback_counts('LGA-disjoint', grouped_partition, grouped_cv)
pd.concat([frozen_fallbacks, grouped_fallbacks], ignore_index=True)

,design,fold,missing_rows,lga,region,global
0,frozen stratified,1,3441,236,1257,1948
1,frozen stratified,2,3399,267,1232,1900
2,frozen stratified,3,3400,243,1206,1951
3,frozen stratified,4,3434,259,1218,1957
4,frozen stratified,5,3465,256,1224,1985
5,LGA-disjoint,1,2780,0,9,2771
6,LGA-disjoint,2,2470,0,2470,0
7,LGA-disjoint,3,4864,0,698,4166
8,LGA-disjoint,4,3387,0,1735,1652
9,LGA-disjoint,5,3638,0,1680,1958


## Component crossing and LGA-disjoint sensitivity

Both population-median components improve slightly alone, so two fixed-weight cross-policy votes check their changed error boundaries without refitting. The closest geography-dependent policy also receives one robustness-only LGA-disjoint comparison.

In [6]:
hybrids = pd.read_csv(RUNTIME_DIR / 'hybrid-summary.csv').set_index('hybrid')
grouped = pd.read_csv(RUNTIME_DIR / 'lga-grouped-summary.csv').set_index('policy')
display(hybrids)
display(grouped.loc[:, [
    'label', 'mean_accuracy', 'accuracy_change', 'fold_wins',
    'worst_fold_change', 'repair_recall',
    'xgboost_accuracy', 'random_forest_accuracy', 'passes_gate',
]])
display(grouped_folds)

,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,non_functional_recall
hybrid,,,,,,
baseline XGBoost + population-median Random Forest,0.815867,-0.000379,2,-0.001894,0.348298,0.783449
population-median XGBoost + baseline Random Forest,0.815509,-0.000737,1,-0.001894,0.346561,0.784490


,label,mean_accuracy,accuracy_change,fold_wins,worst_fold_change,repair_recall,xgboost_accuracy,random_forest_accuracy,passes_gate
policy,,,,,,,,,
population_geographic_median,Population geographic median,0.722284,0.001277,4,-0.003057,0.033477,0.717057,0.723640,False
baseline,Accepted numeric policy,0.721007,0.000000,0,0.000000,0.034971,0.716937,0.721917,False


,rows,lgas,functional_share,repair_share,non_functional_share
validation_fold,,,,,
1,9489,22,0.546844,0.069976,0.383181
2,9528,24,0.532955,0.084278,0.382767
3,9634,27,0.546606,0.073801,0.379593
4,9384,25,0.539429,0.067349,0.393223
5,9485,27,0.549499,0.067897,0.382604


## Decision

Retain the accepted numeric policy. Geographic population imputation improves the LGA-disjoint mean by 0.128 points and wins four folds, but loses 0.034 points on the primary folds and breaches the grouped worst-fold limit. Keep it as a named transfer technique, not a competition-policy change. Stop numeric tuning on these folds; move the next bounded data loop to categorical hierarchy ablation, beginning with `management`, `scheme_management` and `management_group`.